In [1]:
"""
Demucs-inspired architecture for guitar distortion removal.
Adapted from Hybrid Demucs (v3) for 1-in / 1-out mono audio.

Original paper: Défossez et al., "Hybrid Spectrogram and Waveform Source Separation"
https://arxiv.org/abs/2111.03600
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [ ]:
# ---------------------------------------------------------------------------
# Utility modules
# ---------------------------------------------------------------------------

class BLSTM(nn.Module):
    """Bidirectional LSTM used at the bottleneck of each branch."""

    def __init__(self, channels: int, num_layers: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=channels,
            hidden_size=channels,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.proj = nn.Linear(channels * 2, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        B, C, T = x.shape
        x = x.permute(0, 2, 1)          # (B, T, C)
        x, _ = self.lstm(x)             # (B, T, 2C)
        x = self.proj(x)                # (B, T, C)
        return x.permute(0, 2, 1)       # (B, C, T)


class CrossDomainTransformer(nn.Module):
    """
    Lightweight cross-domain attention block.
    Allows the time-domain and frequency-domain branches to exchange information.
    """

    def __init__(self, channels: int, num_heads: int = 4):
        super().__init__()
        self.attn_t2f = nn.MultiheadAttention(channels, num_heads, batch_first=True)  # ???
        self.attn_f2t = nn.MultiheadAttention(channels, num_heads, batch_first=True)
        self.norm_t = nn.LayerNorm(channels)
        self.norm_f = nn.LayerNorm(channels)

    def forward(
        self, t_feat: torch.Tensor, f_feat: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            t_feat: time-branch features  (B, C, T_t)
            f_feat: freq-branch features  (B, C, T_f)
        Returns:
            Updated (t_feat, f_feat) after cross-attention.
        """
        # Reshape to (B, T, C) for MultiheadAttention
        t = t_feat.permute(0, 2, 1)
        f = f_feat.permute(0, 2, 1)

        # Time queries freq
        t_out, _ = self.attn_t2f(t, f, f)
        t = self.norm_t(t + t_out)

        # Freq queries time
        f_out, _ = self.attn_f2t(f, t, t)
        f = self.norm_f(f + f_out)

        return t.permute(0, 2, 1), f.permute(0, 2, 1)

In [ ]:
# ---------------------------------------------------------------------------
# Waveform branch (time domain)
# ---------------------------------------------------------------------------

class TimeBranchEncoder(nn.Module):
    """
    Causal strided-conv encoder operating on raw waveform.
    Each layer doubles the channel count and halves the time resolution.
    """

    def __init__(self, in_channels: int = 1, channels: int = 48, depth: int = 5, stride: int = 4):
        super().__init__()
        self.layers = nn.ModuleList()
        ch = in_channels
        for i in range(depth):
            out_ch = channels * (2 ** i)
            self.layers.append(
                nn.Sequential(
                    nn.Conv1d(ch, out_ch, kernel_size=8, stride=stride, padding=2),
                    nn.ReLU(),
                    nn.Conv1d(out_ch, out_ch, kernel_size=1),   # channel mixer
                    nn.GLU(dim=1) if False else nn.Identity(),  # optionally swap to GLU ???
                )
            )
            ch = out_ch
        self.out_channels = ch

    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        """Returns list of skip-connection tensors from each encoder stage."""
        skips = []
        for layer in self.layers:       # Each skips element has one more filter/layer applied
            x = layer(x)
            skips.append(x)
        return skips  # final element is the bottleneck


class TimeBranchDecoder(nn.Module):
    """
    Mirrored transposed-conv decoder with skip connections.
    """

    def __init__(self, channels: int = 48, depth: int = 5, stride: int = 4):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in reversed(range(depth)):
            in_ch = channels * (2 ** i)
            out_ch = channels * (2 ** (i - 1)) if i > 0 else 1
            self.layers.append(
                nn.Sequential(
                    nn.Conv1d(in_ch * 2, in_ch, kernel_size=1),  # fuse skip
                    nn.ReLU(),
                    nn.ConvTranspose1d(in_ch, out_ch, kernel_size=8, stride=stride, padding=2),
                )
            )

    def forward(self, skips: list[torch.Tensor]) -> torch.Tensor:
        x = skips[-1]
        for i, layer in enumerate(self.layers):     # Pairs most filtered skips element with earlier elements one by one
            skip = skips[-(i + 1)]
            x = torch.cat([x, skip], dim=1)   # channel-wise skip concat
            x = layer(x)
        return x  # (B, 1, T)

In [ ]:
# ---------------------------------------------------------------------------
# Spectrogram branch (frequency domain)
# ---------------------------------------------------------------------------

class FreqBranchEncoder(nn.Module):
    """
    2-D conv encoder operating on complex STFT magnitude/phase.
    Treats the spectrogram as a (freq, time) image.
    """

    def __init__(self, in_channels: int = 2, channels: int = 48, depth: int = 4):
        super().__init__()
        self.layers = nn.ModuleList()
        ch = in_channels
        for i in range(depth):
            out_ch = channels * (2 ** i)
            self.layers.append(
                nn.Sequential(
                    nn.Conv2d(ch, out_ch, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1)),
                    nn.ReLU(),
                )
            )
            ch = out_ch
        self.out_channels = ch

    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        """x: (B, 2, F, T) — real and imaginary STFT channels."""
        skips = []
        for layer in self.layers:
            x = layer(x)
            skips.append(x)
        return skips


class FreqBranchDecoder(nn.Module):
    """
    Mirrored 2-D transposed-conv decoder.
    """

    def __init__(self, channels: int = 48, depth: int = 4):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in reversed(range(depth)):
            in_ch = channels * (2 ** i)
            out_ch = channels * (2 ** (i - 1)) if i > 0 else 2
            self.layers.append(
                nn.Sequential(
                    nn.Conv2d(in_ch * 2, in_ch, kernel_size=1),
                    nn.ReLU(),
                    nn.ConvTranspose2d(in_ch, out_ch, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1)),
                )
            )

    def forward(self, skips: list[torch.Tensor]) -> torch.Tensor:
        x = skips[-1]
        for i, layer in enumerate(self.layers):
            skip = skips[-(i + 1)]
            x = torch.cat([x, skip], dim=1)
            x = layer(x)
        return x  # (B, 2, F, T) — predicted real/imag


# ---------------------------------------------------------------------------
# Main Demucs model
# ---------------------------------------------------------------------------

class GuitarDemucs(nn.Module):
    """
    Hybrid Demucs adapted for mono guitar distortion removal.

    Signal flow:
      Input waveform
          ├─── Time encoder → BLSTM → cross-attn ←──┐
          │                                           │ (exchange)
          └─── STFT → Freq encoder → BLSTM → cross-attn
                                                      │
          Time decoder (+ skip connections) ──────────┤
          Freq decoder (+ skip connections) ──────────┘
              │                  │
          waveform out    iSTFT out
              └──────────┬───────┘
                    Sum → clean waveform
    """

    def __init__(
        self,
        sample_rate: int = 44100,
        channels: int = 48,           # base channel count (scale down for smaller model)
        time_depth: int = 5,          # encoder/decoder depth for time branch
        freq_depth: int = 4,          # encoder/decoder depth for freq branch
        time_stride: int = 4,
        n_fft: int = 4096,
        hop_length: int = 1024,
        lstm_layers: int = 2,
        transformer_heads: int = 4,
    ):
        super().__init__()

        self.sample_rate = sample_rate
        self.n_fft = n_fft
        self.hop_length = hop_length

        # ── Time branch ──────────────────────────────────────────────────────
        self.time_encoder = TimeBranchEncoder(
            in_channels=1, channels=channels, depth=time_depth, stride=time_stride
        )
        self.time_bottleneck = BLSTM(
            channels=self.time_encoder.out_channels, num_layers=lstm_layers
        )
        self.time_decoder = TimeBranchDecoder(
            channels=channels, depth=time_depth, stride=time_stride
        )

        # ── Freq branch ──────────────────────────────────────────────────────
        self.freq_encoder = FreqBranchEncoder(
            in_channels=2, channels=channels, depth=freq_depth
        )
        freq_bottleneck_ch = channels * (2 ** (freq_depth - 1))
        self.freq_bottleneck = BLSTM(
            channels=freq_bottleneck_ch, num_layers=lstm_layers
        )
        self.freq_decoder = FreqBranchDecoder(channels=channels, depth=freq_depth)

        # ── Cross-domain transformer ──────────────────────────────────────────
        # Projects both branches to a common channel dim before cross-attention
        t_ch = self.time_encoder.out_channels
        f_ch = freq_bottleneck_ch
        shared_ch = min(t_ch, f_ch)

        self.t_proj = nn.Conv1d(t_ch, shared_ch, 1)
        self.f_proj = nn.Conv1d(f_ch, shared_ch, 1)  # freq branch flattened over freq dim
        self.cross_attn = CrossDomainTransformer(shared_ch, num_heads=transformer_heads)
        self.t_unproj = nn.Conv1d(shared_ch, t_ch, 1)
        self.f_unproj = nn.Conv1d(shared_ch, f_ch, 1)

        # Final output conv (optional post-processing)
        self.output_conv = nn.Conv1d(1, 1, kernel_size=1)

    # ── STFT helpers ─────────────────────────────────────────────────────────

    def _stft(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T) → complex STFT → (B, 2, F, T_f)  [real, imag]"""
        window = torch.hann_window(self.n_fft, device=x.device)
        stft = torch.stft(
            x, n_fft=self.n_fft, hop_length=self.hop_length,
            window=window, return_complex=True
        )  # (B, F, T_f)
        return torch.stack([stft.real, stft.imag], dim=1)  # (B, 2, F, T_f)

    def _istft(self, x: torch.Tensor, length: int) -> torch.Tensor:
        """x: (B, 2, F, T_f) → waveform (B, T)"""
        window = torch.hann_window(self.n_fft, device=x.device)
        cplx = torch.complex(x[:, 0], x[:, 1])
        return torch.istft(
            cplx, n_fft=self.n_fft, hop_length=self.hop_length,
            window=window, length=length
        )

    # ── Forward pass ─────────────────────────────────────────────────────────

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: distorted mono waveform  (B, 1, T)
        Returns:
            clean waveform estimate     (B, 1, T)
        """
        B, C, T = x.shape
        assert C == 1, "Model expects mono input (1 channel)"

        # ── Time branch encode ────────────────────────────────────────────────
        t_skips = self.time_encoder(x)              # list of (B, C_i, T_i)
        t_bottleneck = self.time_bottleneck(t_skips[-1])

        # ── Freq branch encode ────────────────────────────────────────────────
        stft = self._stft(x.squeeze(1))             # (B, 2, F, T_f)
        f_skips = self.freq_encoder(stft)           # list of (B, C_i, F_i, T_f)
        f_last = f_skips[-1]                        # (B, C_bottleneck, F_reduced, T_f)

        # Flatten freq dim for LSTM: treat (F_reduced * C) or mean-pool
        B2, Cf, Ff, Tf = f_last.shape
        f_flat = f_last.mean(dim=2)                 # (B, C, T_f)  — mean over freq
        f_bottleneck = self.freq_bottleneck(f_flat) # (B, C, T_f)

        # ── Cross-domain attention ────────────────────────────────────────────
        t_proj = self.t_proj(t_bottleneck)          # (B, shared_ch, T_t)
        f_proj = self.f_proj(f_bottleneck)          # (B, shared_ch, T_f)

        # Interpolate to same temporal length for attention
        T_min = min(t_proj.shape[-1], f_proj.shape[-1])
        t_proj = F.adaptive_avg_pool1d(t_proj, T_min)
        f_proj = F.adaptive_avg_pool1d(f_proj, T_min)

        t_proj, f_proj = self.cross_attn(t_proj, f_proj)

        # Unproject back to branch channel dims
        t_bottleneck = t_bottleneck + F.interpolate(
            self.t_unproj(t_proj), size=t_bottleneck.shape[-1]
        )
        f_bottleneck = f_bottleneck + F.interpolate(
            self.f_unproj(f_proj), size=f_bottleneck.shape[-1]
        )

        # ── Time branch decode ────────────────────────────────────────────────
        t_skips[-1] = t_bottleneck
        t_out = self.time_decoder(t_skips)          # (B, 1, T)

        # ── Freq branch decode ────────────────────────────────────────────────
        # Broadcast bottleneck back to (B, C, F_reduced, T_f)
        f_bottleneck_2d = f_bottleneck.unsqueeze(2).expand_as(f_last)
        f_skips[-1] = f_bottleneck_2d
        f_out_stft = self.freq_decoder(f_skips)     # (B, 2, F, T_f)
        f_out = self._istft(f_out_stft, length=T).unsqueeze(1)  # (B, 1, T)

        # ── Combine branches ──────────────────────────────────────────────────
        out = t_out + f_out                         # element-wise sum
        out = self.output_conv(out)

        return out


# ---------------------------------------------------------------------------
# Loss function
# ---------------------------------------------------------------------------

class MultiScaleSTFTLoss(nn.Module):
    """
    Multi-scale spectral loss — essential for audio quality.
    Penalises differences in magnitude spectrogram at multiple resolutions.
    Combine with L1 waveform loss for best results.
    """

    def __init__(self, fft_sizes: list[int] = [2048, 1024, 512, 256]):
        super().__init__()
        self.fft_sizes = fft_sizes

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """pred, target: (B, 1, T)"""
        loss = 0.0
        p = pred.squeeze(1)
        t = target.squeeze(1)
        for n_fft in self.fft_sizes:
            hop = n_fft // 4
            window = torch.hann_window(n_fft, device=pred.device)
            P = torch.stft(p, n_fft, hop, window=window, return_complex=True).abs()
            T = torch.stft(t, n_fft, hop, window=window, return_complex=True).abs()
            loss += F.l1_loss(P, T)
            loss += F.l1_loss(P.log1p(), T.log1p())   # log-magnitude for perceptual quality
        return loss / len(self.fft_sizes)


class CombinedLoss(nn.Module):
    """L1 waveform + multi-scale STFT loss."""

    def __init__(self, stft_weight: float = 0.5):
        super().__init__()
        self.stft_loss = MultiScaleSTFTLoss()
        self.stft_weight = stft_weight

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        l1 = F.l1_loss(pred, target)
        stft = self.stft_loss(pred, target)
        return l1 + self.stft_weight * stft


# ---------------------------------------------------------------------------
# Quick sanity check
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    SAMPLE_RATE = 44100
    DURATION_SEC = 2
    BATCH_SIZE = 2

    model = GuitarDemucs(
        sample_rate=SAMPLE_RATE,
        channels=32,        # reduced for quick test; use 48+ for real training
        time_depth=4,
        freq_depth=3,
    )

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params / 1e6:.2f}M")

    x = torch.randn(BATCH_SIZE, 1, SAMPLE_RATE * DURATION_SEC)
    target = torch.randn_like(x)

    with torch.no_grad():
        out = model(x)

    print(f"Input shape:  {x.shape}")
    print(f"Output shape: {out.shape}")
    assert out.shape == x.shape, "Output shape mismatch!"

    criterion = CombinedLoss()
    loss = criterion(out, target)
    print(f"Loss: {loss.item():.4f}")
    print("Sanity check passed.")